<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/HAARDWT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.environ['KAGGLE_API_TOKEN'] = "KGAT_c03d989b55c966d18c971a92b023645b"

!kaggle datasets download -d britikak/busi-dataset



Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:02<00:00, 69.2MB/s]



In [2]:
!unzip -q busi-dataset.zip -d busi_dataset

In [3]:
import os
import copy
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from PIL import Image
import albumentations as A
from tqdm import tqdm

torch.backends.cudnn.benchmark = True

BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 50

torch.manual_seed(999)
random.seed(999)
np.random.seed(999)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]
                if not mask_files: continue
                self.samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, (mask > 0).astype(np.uint8))
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image, combined_mask = augmented["image"], augmented["mask"]

        return torch.from_numpy(image).permute(2, 0, 1).float(), torch.from_numpy(combined_mask).unsqueeze(0).float()

full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
indices = list(range(len(full_dataset)))
np.random.shuffle(indices)

train_size, val_size = int(0.8 * len(full_dataset)), int(0.1 * len(full_dataset))
train_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=train_transform), indices[val_size:train_size + val_size])
val_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[:val_size])
test_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[train_size + val_size:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [4]:
class HaarDWT(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        ll = torch.tensor([[1., 1.], [1., 1.]]) / 2.0
        lh = torch.tensor([[-1., -1.], [1., 1.]]) / 2.0
        hl = torch.tensor([[-1., 1.], [-1., 1.]]) / 2.0
        hh = torch.tensor([[1., -1.], [-1., 1.]]) / 2.0

        filters = torch.stack([ll, lh, hl, hh], dim=0).unsqueeze(1)
        filters = filters.repeat(in_channels, 1, 1, 1)

        self.register_buffer('filters', filters)
        self.in_channels = in_channels

    def forward(self, x):
        return F.conv2d(x, self.filters, stride=2, groups=self.in_channels)

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

In [5]:
class WaveletResNet50UNet(nn.Module):
    def __init__(self, out_channels=1):
        super().__init__()

        # PATH A: ResNet50 (Semantic Context)
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.r_e1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.r_pool = resnet.maxpool
        self.r_e2 = resnet.layer1
        self.r_e3 = resnet.layer2
        self.r_e4 = resnet.layer3
        self.r_bottleneck = resnet.layer4

        # PATH B: Haar Wavelet Cascade (Edge Detail)
        self.w_dwt1 = HaarDWT(in_channels=3)
        self.w_conv1 = DoubleConv(3 * 4, 64)

        self.w_dwt2 = HaarDWT(in_channels=64)
        self.w_conv2 = DoubleConv(64 * 4, 256)

        self.w_dwt3 = HaarDWT(in_channels=256)
        self.w_conv3 = DoubleConv(256 * 4, 512)

        self.w_dwt4 = HaarDWT(in_channels=512)
        self.w_conv4 = DoubleConv(512 * 4, 1024)

        self.w_dwt_b = HaarDWT(in_channels=1024)
        self.w_conv_b = DoubleConv(1024 * 4, 2048)

        # --- DECODER ---
        self.up4 = nn.ConvTranspose2d(2048 * 2, 1024, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024 + 2048, 1024)

        self.up3 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512 + 1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256 + 512, 256)

        self.up1 = nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64 + 128, 64)

        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec0 = DoubleConv(32, 32)
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        # 1. PARALLEL ENCODING
        re1 = self.r_e1(x)
        we1 = self.w_conv1(self.w_dwt1(x))

        re2 = self.r_e2(self.r_pool(re1))
        we2 = self.w_conv2(self.w_dwt2(we1))

        re3 = self.r_e3(re2)
        we3 = self.w_conv3(self.w_dwt3(we2))

        re4 = self.r_e4(re3)
        we4 = self.w_conv4(self.w_dwt4(we3))

        rb = self.r_bottleneck(re4)
        wb = self.w_conv_b(self.w_dwt_b(we4))

        # 2. FUSED BOTTLENECK
        b_fused = torch.cat([rb, wb], dim=1)

        # 3. FUSED SKIP DECODER
        s4_fused = torch.cat([re4, we4], dim=1)
        d4 = self.dec4(torch.cat([s4_fused, self.up4(b_fused)], dim=1))

        s3_fused = torch.cat([re3, we3], dim=1)
        d3 = self.dec3(torch.cat([s3_fused, self.up3(d4)], dim=1))

        s2_fused = torch.cat([re2, we2], dim=1)
        d2 = self.dec2(torch.cat([s2_fused, self.up2(d3)], dim=1))

        s1_fused = torch.cat([re1, we1], dim=1)
        d1 = self.dec1(torch.cat([s1_fused, self.up1(d2)], dim=1))

        d0 = self.dec0(self.up0(d1))
        return self.final_conv(d0)

In [6]:
class StrictBCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        preds = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        inter = (preds * targets_f).sum()
        dice_loss = 1 - (2 * inter + self.smooth) / (preds.sum() + targets_f.sum() + self.smooth)
        return 0.2 * bce_loss + 0.8 * dice_loss

def strict_dice_coef(y_true, logits, smooth=1e-5):
    y_pred = (torch.sigmoid(logits) > 0.5).float().view(-1)
    y_true_f = y_true.view(-1)
    inter = (y_true_f * y_pred).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred.sum() + smooth)

In [8]:
# Initialize Model
model = WaveletResNet50UNet(out_channels=1).to(device)

criterion = StrictBCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

# Trackers
best_val_dice = 0.0
best_model_weights = None
best_epoch = 0

print("===================================================")
print(f" Initializing Training: Dual-Path Wavelet-ResNet")
print(f"===================================================")

for epoch in range(EPOCHS):
    # --- 1. TRAINING PHASE ---
    model.train()
    train_loss = train_dice = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        with torch.no_grad():
            train_dice += strict_dice_coef(masks, logits).item()

    scheduler.step()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_dice = train_dice / len(train_loader)

    # --- 2. VALIDATION PHASE ---
    model.eval()
    val_loss = val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)

            val_loss += loss.item()
            val_dice += strict_dice_coef(masks, logits).item()

    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = val_dice / len(val_loader)

    # --- 3. EPOCH DASHBOARD ---
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        best_epoch = epoch + 1
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_wavelet_resnet.pth")


print("\n===================================================")
print(" Training Complete. Commencing Unseen Test Data ")
print(f" Extracting  Weights from Epoch: {best_epoch} (Val Dice: {best_val_dice:.4f}) ")
print("===================================================")


model.load_state_dict(best_model_weights)
model.eval()

test_loss = test_dice = 0

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        test_loss += loss.item()
        test_dice += strict_dice_coef(masks, logits).item()

avg_test_loss = test_loss / len(test_loader)
avg_test_dice = test_dice / len(test_loader)

print(f"\n FINAL TEST DICE SCORE: {avg_test_dice:.4f}")
print(f" FINAL TEST LOSS: {avg_test_loss:.4f}")

 Initializing Training: Dual-Path Wavelet-ResNet


Epoch 1/50: 100%|██████████| 130/130 [00:35<00:00,  3.68it/s]


Epoch [1/50] | Train Dice: 0.5016 | Val Dice: 0.6131 | Train Loss: 0.6712 | Val Loss: 0.5891


Epoch 2/50: 100%|██████████| 130/130 [00:34<00:00,  3.77it/s]


Epoch [2/50] | Train Dice: 0.6092 | Val Dice: 0.7003 | Train Loss: 0.5833 | Val Loss: 0.5253


...

Epoch [48/50] | Train Dice: 0.8792 | Val Dice: 0.7539 | Train Loss: 0.1225 | Val Loss: 0.2550


Epoch 49/50: 100%|██████████| 130/130 [00:34<00:00,  3.74it/s]


Epoch [49/50] | Train Dice: 0.8800 | Val Dice: 0.7523 | Train Loss: 0.1208 | Val Loss: 0.2557


Epoch 50/50: 100%|██████████| 130/130 [00:34<00:00,  3.73it/s]


Epoch [50/50] | Train Dice: 0.8822 | Val Dice: 0.7487 | Train Loss: 0.1202 | Val Loss: 0.2586

 Training Complete. Commencing Unseen Test Data 
 Extracting  Weights from Epoch: 18 (Val Dice: 0.7688) 

 FINAL TEST DICE SCORE: 0.7587
 FINAL TEST LOSS: 0.2432
